# 光谱处理：DASH 分类 & 膨胀速度测量

本 notebook 对下载的 WISeREP 光谱进行：
1. **可视化** — 展示原始光谱
2. **DASH 分类** — 用 [astrodash](https://github.com/daniel-murray/astrodash) 进行模板匹配，获取超新星类型、年龄和红移
3. **膨胀速度** — 基于 P-Cygni 吸收谷的蓝移测量抛射物速度

> 运行前请确保 `astro_env` 环境已创建，并用 `python scripts/fetch_aux_data.py` 下载了光谱数据。

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys, io, warnings
from pathlib import Path
from scipy.signal import savgol_filter
sys.path.append(os.path.abspath('..'))
from src.wiserep import plot_spectra

## 1. 加载并可视化光谱

从 `output/<target>/spectrum/` 读取 `.dat` 文件并绘图。修改 `target_name` 可切换目标。

In [ ]:
target_name = 'SN2026jlm'
filedir = f'../data/{target_name}'
filepath = list(Path(filedir).glob('*.txt'))[1:]

plot_spectra(filepath, target_name, jupyter=True)

## 2. DASH 分类

使用 DASH 深度学习模型对光谱进行模板匹配。需要指定宿主星系红移 `host_galaxy_z`（可从 TNS catalog 查询，填 `None` 则是未知红移模式）。

支持谱线：
| 类型 | 特征线 | 静止波长 |
|------|--------|----------|
| Ia | Si II | 6355.0 Å |
| II / IIn | Hα | 6562.8 Å |
| Ibc | He I | 5875.6 Å |

In [ ]:
import astrodash

# 抑制 tensorflow / pandas 的冗长输出
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore')

filename_str = [str(f) for f in filepath]
print(f'📂 光谱文件: {filename_str}')
print(f'✅ 文件存在: {Path(filename_str[0]).exists()}')
print("正在启动 DASH 进行模板匹配和深度学习分类...")

host_galaxy_z = 0.016738  # 填 None 则为未知红移模式

try:
    _orig_stdout = sys.stdout
    sys.stdout = io.StringIO()
    classification = astrodash.Classify(
        filenames=filename_str,
        knownZ=host_galaxy_z,
    )
    ret = classification.list_best_matches()
    sys.stdout = _orig_stdout

    # ret: (bestMatchLists, redshifts, bestBroadTypes, rlapLabels, reliable, redshiftErrs)
    best_fits = ret[2]
    redshifts = ret[1]

    if best_fits is not None and len(best_fits) > 0:
        best_match = best_fits[0]
        sn_name = best_match[0]    # 模板名 (如 Ia-norm, IIP)
        sn_type = best_match[1]    # 年龄范围
        sn_age  = best_match[2]    # 概率

        if host_galaxy_z is not None and redshifts is not None and len(redshifts) > 0:
            fitted_z = float(redshifts[0]) if redshifts.ndim == 1 else float(redshifts[0, 0])
            sn_z = fitted_z if abs(fitted_z) > 1e-6 else host_galaxy_z
        else:
            sn_z = None

        print("-" * 40)
        print(f"🎯 DASH 分类结果 (已知红移 z={host_galaxy_z}):")
        print(f"类型 (Type):         {sn_type}")
        print(f"年龄 (Age):          {sn_age} days")
        print(f"红移 (z):            {sn_z:.4f}")
        print(f"最佳模板:            {sn_name}")
        print("-" * 40)

        # ── 对比图：观测光谱 vs 最佳模板 ──
        _, _, _, inputImages, _ = classification._input_spectra_info()
        observed = inputImages[0]
        top_template = ret[0][0][0]   # (host, name, age, prob)
        tmpl_name = top_template[1]
        tmpl_age  = top_template[2]
        tmpl_prob = float(top_template[3])
        tmpl_flux = classification.snTemplates[tmpl_name][tmpl_age]["snInfo"][0][1]

        plt.figure(figsize=(10, 6))
        plt.plot(classification.wave, observed, "k-", lw=0.8, label="Observed (rebin)")
        npts = min(len(observed), len(tmpl_flux))
        plt.plot(classification.wave[:npts], tmpl_flux[:npts], "r-", lw=1.0, alpha=0.7,
                 label=f"Template: {tmpl_name} ({tmpl_age})")
        plt.ylabel("Normalized Flux")
        plt.title(f"DASH Fit: {sn_name} ({sn_type})  z={sn_z:.4f}")
        plt.legend(fontsize=9)
        plt.show()
    else:
        print("⚠ DASH 未能找到可靠的匹配结果。")

except Exception as e:
    try: sys.stdout = _orig_stdout
    except: pass
    print(f"❌ DASH 分类失败: {e}")

## 3. 膨胀速度测量

基于 P-Cygni 吸收谷的蓝移计算抛射物膨胀速度。默认使用 DASH 分类结果自动选择特征谱线，也可手动覆盖参数。

### 手动覆盖

修改下面几个变量即可：
```python
sn_z = 0.017          # 手动指定红移
rest_wave = 6562.8    # 手动指定静止波长 (Å)
line_name = "Hα"      # 手动指定谱线名
```

In [ ]:
# 用 DASH 结果，未运行时默认设为 II 型 + Hα
if "sn_z" not in dir() or sn_z is None:
    sn_z = 0.0
    print("⚠ sn_z 未定义，已设为 0 (请先运行 DASH 或手动指定)")

if "sn_name" not in dir() or sn_name is None:
    sn_name = "II"

# 根据 DASH 分类名推断类型，自动选择特征谱线
known_lines = {
    "Hα":   6562.8, "Hβ":   4861.3,
    "Si II": 6355.0, "He I":  5875.6, "Ca II": 3933.7,
}
dash_type = sn_name if sn_name else ""
if dash_type.lower().startswith("ia"):
    rest_wave = known_lines["Si II"]; line_name = "Si II"
elif dash_type.lower().startswith("ii"):
    rest_wave = known_lines["Hα"];    line_name = "Hα"
elif "ib" in dash_type.lower() or "ic" in dash_type.lower():
    rest_wave = known_lines["He I"];  line_name = "He I"
else:
    rest_wave = known_lines["Hα"];    line_name = "Hα"

# ═══ 计算膨胀速度 ═══
spec_file = filepath[0]
data = np.loadtxt(spec_file)
wave = data[:, 0]; flux = data[:, 1]
flux_smoothed = savgol_filter(flux, 15, polyorder=3)

c = 299792.458                                  # 光速 km/s
wave_rest = wave / (1 + sn_z)                   # 去红移
search_half = 400

mask = (wave_rest > rest_wave - search_half) & (wave_rest < rest_wave + search_half)
local_wave = wave_rest[mask]
local_flux = flux_smoothed[mask]

# 在静止波长蓝侧搜索吸收谷
blue_mask = local_wave < rest_wave
if blue_mask.sum() > 5:
    min_idx = np.argmin(local_flux[blue_mask])
    obs_abs_wave = local_wave[blue_mask][min_idx]
else:
    obs_abs_wave = local_wave[np.argmin(local_flux)]

v_exp = c * (rest_wave - obs_abs_wave) / rest_wave

print(f"光谱文件: {spec_file.name}  ({len(wave)} pts, {wave[0]:.0f}-{wave[-1]:.0f} Å)")
print("-" * 50)
print(f"  谱线:      {line_name} (rest λ₀ = {rest_wave} A)")
print(f"  测量蓝移:  λ_obs = {obs_abs_wave:.1f} A")
print(f"  红移:      z = {sn_z:.4f}")
print(f"  🚀 膨胀速度: v_exp = {v_exp:.0f} km/s")
print("-" * 50)

# 绘图
plt.figure(figsize=(10, 6))
plt.plot(wave_rest, flux_smoothed, "b-", lw=0.8, label="Smoothed spectrum")
plt.axvspan(rest_wave - search_half, rest_wave + search_half,
            color="yellow", alpha=0.2, label="Search region")
plt.axvline(rest_wave, color="red", ls="--", lw=1, label=f"{line_name} rest")
plt.axvline(obs_abs_wave, color="green", ls=":", lw=1.5, label=f"Abs. minimum")
plt.xlabel("Rest-frame Wavelength (A)")
plt.ylabel("Flux")
plt.title(f"{target_name} — {line_name} λ{rest_wave:.0f}  v={v_exp:.0f} km/s")
plt.legend(fontsize=8)
plt.show()